<div align='center'>
  <img src='https://img.shields.io/badge/NIAT%20Assignment-Masterclass_3-orange?style=for-the-badge' />
  <h1>🎨 AI Product Image Generator with Gradio</h1>
  <p><i>Official Starter Template — Optimized for T4 GPUs</i></p>
</div>

---

### 🎯 Assignment Goal
Design and implement a complete AI-powered product photography pipeline that transforms simple text into professional-grade visuals.

**Requirements:**
1. **LLM Integration**: Expert prompt engineering.
2. **Diffusion Synthesis**: Studio-quality image generation.
3. **Web Interface**: Interactive Gradio UI.

**⚙️ Hardware Tip:** Enable GPU acceleration (**T4 x2**) in Settings before starting.

## 🛠️ Step 0: Environment & Verification
We initialize the environment, verify GPU availability, and install the core generative AI stack.

In [ ]:
# Verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected! Enable GPU in notebook settings.")

In [ ]:
# Install dependencies
!pip install -q transformers diffusers accelerate gradio sentence-transformers Pillow
print("All dependencies installed.")

In [ ]:
# Create output directories
import os
os.makedirs("images", exist_ok=True)
print("Output directories ready.")

---
## Step 1: Load the Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("PASTE THE PRODUCT DESCRIPTIONS.CSV FILE PATH HERE")
print(f"Loaded {len(df)} product descriptions")
df.head()

import pandas as pd

df = pd.read_csv('starter_notebook.ipynb.csv')
print(f'Loaded {len(df)} product descriptions')
df.head()

In [ ]:
# ============================================================
# TODO: Load your LLM model and tokenizer here
# ============================================================
# Example:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

# YOUR CODE HERE


In [ ]:
# ============================================================
# TODO: Write your prompt engineering function
# ============================================================
# This function should:
# 1. Take a simple product description as input
# 2. Use the LLM to rewrite it into a detailed image generation prompt
# 3. Return the engineered prompt as a string
#
# Tips:
# - Use a good system prompt (role + context + instruction)
# - Keep output under 60 words (CLIP token limit is 77)
# - Include details about lighting, style, composition

def engineer_prompt(product_description):
    # YOUR CODE HERE
    pass

# Test it
# test = engineer_prompt("red wireless headphones on a white background")
# print(test)

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

LLM_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

print(f'📦 Loading LLM: {LLM_MODEL_ID} ...')
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto' if DEVICE == 'cuda' else None,
    trust_remote_code=True,
)
llm_model.eval()
print('✅ LLM loaded.')

In [ ]:
SYSTEM_PROMPT = """You are a professional product photographer and prompt engineer specializing in e-commerce imagery.
Your task: convert a simple product description into a concise, high-quality Stable Diffusion prompt.

Rules:
- Output ONLY the prompt, no explanations or extra text.
- Keep the prompt under 60 words.
- Always include: product type, material/texture, studio lighting, clean background, camera angle, and quality keywords.
- Always include these CLIP-aligned keywords: 'product photography', 'studio lighting', 'white background', 'sharp focus', 'ultra realistic', '8k'.
- Use a consistent structure: [product] [details] [lighting] [background] [camera] [quality].
- Do NOT include negative prompts, special tokens, or markdown formatting."""

def engineer_prompt(product_description):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Product: {product_description}'},
    ]

    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = llm_tokenizer(text, return_tensors='pt').to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    prompt = llm_tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    # Clean up
    prompt = prompt.strip('"').strip("'").strip()
    for prefix in ['Prompt:', 'prompt:', 'Output:', 'output:']:
        if prompt.startswith(prefix):
            prompt = prompt[len(prefix):].strip()

    # Ensure keywords
    clip_keywords = ['product photography', 'studio lighting', 'sharp focus']
    for kw in clip_keywords:
        if kw.lower() not in prompt.lower():
            prompt += f', {kw}'

    return prompt

# Test it
test_val = engineer_prompt('red leather handbag')
print(f'Test prompt: {test_val}')

In [ ]:
# ============================================================
# TODO: Write your image generation function
# ============================================================
# This function should:
# 1. Take an engineered prompt as input
# 2. Generate an image using Stable Diffusion
# 3. Return a PIL Image

def generate_image(engineered_prompt):
    # YOUR CODE HERE
    pass

# Test it
# img = generate_image("A professional photo of red headphones on white background")
# display(img)

## 📊 Step 1: Data Acquisition
Loading the official evaluation dataset containing 15 high-potential product categories.

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

SD_MODEL_ID = 'runwayml/stable-diffusion-v1-5'

print(f'📦 Loading Stable Diffusion: {SD_MODEL_ID} ...')
sd_pipe = StableDiffusionPipeline.from_pretrained(
    SD_MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
sd_pipe.scheduler = DPMSolverMultistepScheduler.from_config(sd_pipe.scheduler.config)
sd_pipe = sd_pipe.to(DEVICE)

# Memory optimizations
if DEVICE == 'cuda':
    sd_pipe.enable_attention_slicing()
    try:
        sd_pipe.enable_xformers_memory_efficient_attention()
        print('✓ xformers enabled')
    except Exception:
        print('⚠ xformers not available')

print('✅ Stable Diffusion pipeline ready.')

NEGATIVE_PROMPT = (
    'blurry, low quality, distorted, deformed, ugly, noisy, text, watermark, '
    'oversaturated, cartoon, illustration, painting, sketch, bad proportions, '
    'cropped, out of frame, duplicate, morbid, mutilated'
)

def generate_image(engineered_prompt):
    generator = torch.Generator(device=DEVICE).manual_seed(42)
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
        result = sd_pipe(
            prompt=engineered_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30,
            guidance_scale=7.5,
            width=512,
            height=512,
            generator=generator,
        )
    return result.images[0]

# Test it
img = generate_image('A professional photo of red headphones on white background')
display(img)

In [ ]:
# ============================================================
# TODO: Generate engineered prompts and images for all 15 products
# ============================================================
# This cell should:
# 1. Loop through all 15 product descriptions
# 2. Engineer a prompt for each
# 3. Generate an image for each
# 4. Save prompts to submission.csv
# 5. Save images as img_1.png through img_15.png in images/ folder


# Save submission CSV
# submission_df = pd.DataFrame(results)
# submission_df.to_csv("submission.csv", index=False)
# print("submission.csv saved!")
# print(f"Images saved in images/ folder")

## 🧠 Step 2: Intelligent Prompt Engineering
Translating user-friendly descriptions into rich, descriptive prompts for Stable Diffusion using `Qwen2.5`.

In [ ]:
import gradio as gr

def gradio_generate(description):
    if not description or not description.strip():
        return 'Please enter a description.', None
    prompt = engineer_prompt(description.strip())
    image = generate_image(prompt)
    return prompt, image

with gr.Blocks(title='AI Product Image Generator') as demo:
    gr.Markdown('# 🎨 AI Product Image Generator')
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label='Product Description', placeholder='e.g. red vintage camera')
            btn = gr.Button('Generate', variant='primary')
        with gr.Column():
            output_prompt = gr.Textbox(label='Engineered Prompt', interactive=False)
            output_image = gr.Image(label='Generated Image', type='pil')

    btn.click(fn=gradio_generate, inputs=input_text, outputs=[output_prompt, output_image])

# To launch the interface, uncomment and run:
# demo.launch(inline=True)

In [ ]:
import base64

encoded = b"IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFVkFMVUFUSU9OIENFTEwgMjogSW1hZ2UgUXVhbGl0eSAoNDAlIHdlaWdodCkKIyBETyBOT1QgTU9ESUZZCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CmltcG9ydCB0b3JjaApmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQ0xJUFByb2Nlc3NvciwgQ0xJUE1vZGVsCmZyb20gUElMIGltcG9ydCBJbWFnZQppbXBvcnQgbnVtcHkgYXMgbnAKCnByaW50KCI9IiAqIDUwKQpwcmludCgiICBFVkFMVUFUSU5HIElNQUdFIFFVQUxJVFkiKQpwcmludCgiPSIgKiA1MCkKCmNsaXBfbW9kZWwgPSBDTElQTW9kZWwuZnJvbV9wcmV0cmFpbmVkKCJvcGVuYWkvY2xpcC12aXQtYmFzZS1wYXRjaDMyIikKY2xpcF9wcm9jZXNzb3IgPSBDTElQUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZCgib3BlbmFpL2NsaXAtdml0LWJhc2UtcGF0Y2gzMiIpCgpkZl9ldmFsID0gcGQucmVhZF9jc3YoImh0dHBzOi8vczMuYXAtc291dGgtMS5hbWF6b25hd3MuY29tL25ldy1hc3NldHMuY2NicC5pbi9mcm9udGVuZC9jb250ZW50L2FpbWwvTWFzdGVyY2xhc3NfTklBVC9wcm9kdWN0X2Rlc2NyaXB0aW9ucy5jc3YiKQoKaW1hZ2Vfc2NvcmVzID0ge30KZm9yIF8sIHJvdyBpbiBkZl9ldmFsLml0ZXJyb3dzKCk6CiAgICBwaWQgPSByb3dbImlkIl0KICAgIGRlc2MgPSByb3dbInByb2R1Y3RfZGVzY3JpcHRpb24iXQogICAgaW1nX3BhdGggPSBmImltYWdlcy9pbWdfe3BpZH0ucG5nIgogICAgCiAgICB0cnk6CiAgICAgICAgaW1hZ2UgPSBJbWFnZS5vcGVuKGltZ19wYXRoKS5jb252ZXJ0KCJSR0IiKQogICAgICAgIGlucHV0cyA9IGNsaXBfcHJvY2Vzc29yKHRleHQ9W2Rlc2NdLCBpbWFnZXM9aW1hZ2UsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9VHJ1ZSkKICAgICAgICAKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0cHV0cyA9IGNsaXBfbW9kZWwoKippbnB1dHMpCiAgICAgICAgICAgIHJhd19zY29yZSA9IG91dHB1dHMubG9naXRzX3Blcl9pbWFnZS5pdGVtKCkKICAgICAgICAKICAgICAgICBub3JtYWxpemVkID0gZmxvYXQobnAuY2xpcCgocmF3X3Njb3JlIC0gMTUuMCkgLyAyMC4wLCAwLjAsIDEuMCkpCiAgICAgICAgaW1hZ2Vfc2NvcmVzW3BpZF0gPSByb3VuZChub3JtYWxpemVkLCA0KQogICAgICAgIHByaW50KGYiICBQcm9kdWN0IHtwaWQ6MmR9OiBDTElQID0ge3Jhd19zY29yZTouMmZ9LCBpbWFnZV9zY29yZSA9IHtub3JtYWxpemVkOi40Zn0iKQogICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgIHByaW50KGYiICBQcm9kdWN0IHtwaWQ6MmR9OiBJTUFHRSBOT1QgRk9VTkQgKHNjb3JlID0gMCkiKQogICAgICAgIGltYWdlX3Njb3Jlc1twaWRdID0gMC4wCgphdmdfaW1hZ2UgPSBucC5tZWFuKGxpc3QoaW1hZ2Vfc2NvcmVzLnZhbHVlcygpKSkKcHJpbnQoZiJcbiAgQXZlcmFnZSBJbWFnZSBTY29yZToge2F2Z19pbWFnZTouNGZ9IikKcHJpbnQoIj0iICogNTAp"  # base64 of: print("Hello, World!")

exec(base64.b64decode(encoded))

In [ ]:
import os
import time

results = []
os.makedirs('images', exist_ok=True)

print('🚀 Starting Batch Processing for 15 Products...')
start_time = time.time()

for idx, row in df.iterrows():
    description = row['product_description']
    pid = int(row['id'])
    
    print(f'[{pid}/15] Processing: {description}')
    
    # 1. Engineer Prompt
    prompt = engineer_prompt(description)
    
    # 2. Generate Image
    image = generate_image(prompt)
    
    # 3. Save Image (Constraint: must be images/img_X.png)
    image_filename = f'img_{pid}.png'
    image_path = os.path.join('images', image_filename)
    image.save(image_path)
    
    results.append({
        'id': pid,
        'engineered_prompt': prompt
    })

# 4. Save submission.csv
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False)

end_time = time.time()
print(f'\n✅ Batch Processing Complete! Total time: {end_time - start_time:.1f}s')
print('📄 submission.csv saved!')
print("🖼️  Images saved in 'images/' folder.")

In [ ]:
import base64

encoded = b"IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBFVkFMVUFUSU9OIENFTEwgNDogR2VuZXJhdGUgZmluYWxfc2NvcmVzLmNzdiBmb3IgbGVhZGVyYm9hcmQKIyBETyBOT1QgTU9ESUZZCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgQ29tYmluZWQgc2NvcmUgcGVyIHByb2R1Y3Q6CiMgc2NvcmUgPSBwcm9tcHRfc2NvcmUgKiAwLjQgKyBpbWFnZV9zY29yZSAqIDAuNCArIGdyYWRpb19zY29yZSAqIDAuMgojIFBlcmZlY3Qgc2NvcmUgPSAxLjAgcGVyIHByb2R1Y3QuIExlYWRlcmJvYXJkIHVzZXMgUk1TRSAobG93ZXIgPSBiZXR0ZXIpLgoKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgbnVtcHkgYXMgbnAKCnJvd3MgPSBbXQpmb3IgcGlkIGluIHJhbmdlKDEsIDE2KToKICAgIHAgPSBwcm9tcHRfc2NvcmVzLmdldChwaWQsIDAuMCkKICAgIGkgPSBpbWFnZV9zY29yZXMuZ2V0KHBpZCwgMC4wKQogICAgZyA9IGdyYWRpb192YWwKICAgIGNvbWJpbmVkID0gcm91bmQocCAqIDAuNCArIGkgKiAwLjQgKyBnICogMC4yLCA0KQogICAgcm93cy5hcHBlbmQoeyJpZCI6IHBpZCwgInNjb3JlIjogY29tYmluZWR9KQoKZmluYWwgPSBwZC5EYXRhRnJhbWUocm93cykKZmluYWwudG9fY3N2KCJmaW5hbF9zY29yZXMuY3N2IiwgaW5kZXg9RmFsc2UpCgpwcmludCgiPSIgKiA1MCkKcHJpbnQoIiAgICAgICAgIEZJTkFMIEVWQUxVQVRJT04gUkVTVUxUUyIpCnByaW50KCI9IiAqIDUwKQpwcmludCgpCmZvciBfLCByIGluIGZpbmFsLml0ZXJyb3dzKCk6CiAgICBwcmludChmIiAgUHJvZHVjdCB7aW50KHJbJ2lkJ10pOjJkfTogc2NvcmUgPSB7clsnc2NvcmUnXTouNGZ9IikKcHJpbnQoKQoKYXZnX3Byb21wdCA9IG5wLm1lYW4obGlzdChwcm9tcHRfc2NvcmVzLnZhbHVlcygpKSkKYXZnX2ltYWdlID0gbnAubWVhbihsaXN0KGltYWdlX3Njb3Jlcy52YWx1ZXMoKSkpCgpwcmludChmIiAgUHJvbXB0IFF1YWxpdHkgKGF2ZykgOiB7YXZnX3Byb21wdDouNGZ9IikKcHJpbnQoZiIgIEltYWdlIFF1YWxpdHkgIChhdmcpIDoge2F2Z19pbWFnZTouNGZ9IikKcHJpbnQoZiIgIEdyYWRpbyBVSSAgICAgICAgICAgIDoge2dyYWRpb192YWw6LjFmfSIpCnByaW50KCkKcHJpbnQoZiIgIFByb21wdCBQb2ludHMgIDoge2F2Z19wcm9tcHQgKiA0MDo1LjFmfSAvIDQwIikKcHJpbnQoZiIgIEltYWdlIFBvaW50cyAgIDoge2F2Z19pbWFnZSAqIDQwOjUuMWZ9IC8gNDAiKQpwcmludChmIiAgR3JhZGlvIFBvaW50cyAgOiB7Z3JhZGlvX3ZhbCAqIDIwOjUuMWZ9IC8gMjAiKQp0b3RhbCA9IGF2Z19wcm9tcHQgKiA0MCArIGF2Z19pbWFnZSAqIDQwICsgZ3JhZGlvX3ZhbCAqIDIwCnByaW50KGYiICB7J+KUgCcgKiAzMH0iKQpwcmludChmIiAgVE9UQUwgU0NPUkUgICAgOiB7dG90YWw6NS4xZn0gLyAxMDAiKQpwcmludCgpCnByaW50KCI9IiAqIDUwKQpwcmludCgiICBmaW5hbF9zY29yZXMuY3N2IHNhdmVkLiIpCnByaW50KCIgIFN1Ym1pdCB0aGlzIGZpbGUgdG8gdGhlIGNvbXBldGl0aW9uIGxlYWRlcmJvYXJkLiIpCnByaW50KCI9IiAqIDUwKQ=="  # base64 of: print("Hello, World!")

exec(base64.b64decode(encoded))